In [ ]:
# sentiment_analysis_lstm.py
# Sentiment Analysis on IMDB using LSTM Neural Network

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout, BatchNormalization, SpatialDropout1D
)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
import warnings
warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────
# 1. HYPERPARAMETERS
# ─────────────────────────────────────────────
VOCAB_SIZE   = 10_000   # keep top 10k most frequent words
MAX_LENGTH   = 200      # pad/truncate each review to 200 tokens
EMBED_DIM    = 128      # embedding vector size
LSTM1_UNITS  = 128
LSTM2_UNITS  = 64
DENSE_UNITS  = 64
DROPOUT_RATE = 0.4
BATCH_SIZE   = 64
EPOCHS       = 10
LR           = 1e-3


# ─────────────────────────────────────────────
# 2. LOAD & PREPROCESS DATA
# ─────────────────────────────────────────────
print("Loading IMDB dataset...")
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# Pad sequences to uniform length
X_train = pad_sequences(X_train, maxlen=MAX_LENGTH, padding="post", truncating="post")
X_test  = pad_sequences(X_test,  maxlen=MAX_LENGTH, padding="post", truncating="post")

print(f"Training samples : {X_train.shape}")
print(f"Test samples     : {X_test.shape}")
print(f"Positive (train) : {y_train.sum()} / {len(y_train)}")


# ─────────────────────────────────────────────
# 3. BUILD THE LSTM MODEL
# ─────────────────────────────────────────────
def build_model(vocab_size, max_len, embed_dim,
                lstm1, lstm2, dense, dropout, lr):
    model = Sequential([
        # Word embeddings: integer tokens → dense vectors
        Embedding(vocab_size, embed_dim, input_length=max_len),

        # Spatial dropout on the embedding (drops entire feature maps)
        SpatialDropout1D(0.2),

        # First LSTM layer – return full sequence for stacking
        LSTM(lstm1, return_sequences=True, dropout=dropout/2,
             recurrent_dropout=dropout/4),
        Dropout(dropout),

        # Second LSTM layer – returns final hidden state only
        LSTM(lstm2, dropout=dropout/2, recurrent_dropout=dropout/4),
        Dropout(dropout),

        # Fully-connected head
        Dense(dense, activation="relu"),
        BatchNormalization(),
        Dropout(dropout - 0.1),

        # Binary output
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


model = build_model(VOCAB_SIZE, MAX_LENGTH, EMBED_DIM,
                    LSTM1_UNITS, LSTM2_UNITS, DENSE_UNITS,
                    DROPOUT_RATE, LR)
model.summary()


# ─────────────────────────────────────────────
# 4. CALLBACKS
# ─────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor="val_loss", patience=3,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2,
        min_lr=1e-6, verbose=1
    ),
    ModelCheckpoint(
        "best_model.keras", monitor="val_accuracy",
        save_best_only=True, verbose=0
    )
]


# ─────────────────────────────────────────────
# 5. TRAIN
# ─────────────────────────────────────────────
print("\nTraining model...")
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)


# ─────────────────────────────────────────────
# 6. EVALUATE
# ─────────────────────────────────────────────
print("\n──── Test Set Evaluation ────")
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Loss     : {loss:.4f}")
print(f"Accuracy : {acc:.4f}")

y_proba = model.predict(X_test, verbose=0).ravel()
y_pred  = (y_proba >= 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC  : {auc:.4f}")


# ─────────────────────────────────────────────
# 7. VISUALISATIONS
# ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("LSTM Sentiment Analysis — Results", fontsize=14, fontweight="bold")

# 7a. Training curves
ax = axes[0, 0]
ax.plot(history.history["accuracy"],     label="Train acc")
ax.plot(history.history["val_accuracy"], label="Val acc", linestyle="--")
ax.set_title("Accuracy over epochs")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.plot(history.history["loss"],     label="Train loss")
ax.plot(history.history["val_loss"], label="Val loss", linestyle="--")
ax.set_title("Loss over epochs")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(alpha=0.3)

# 7b. Confusion matrix
ax = axes[1, 0]
cm = confusion_matrix(y_test, y_pred)
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Negative", "Positive"])
ax.set_yticklabels(["Negative", "Positive"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black",
                fontsize=14, fontweight="bold")
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.colorbar(im, ax=ax)

# 7c. ROC curve
ax = axes[1, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax.plot(fpr, tpr, label=f"AUC = {auc:.4f}", color="steelblue")
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
ax.set_title("ROC Curve")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("results.png", dpi=150, bbox_inches="tight")
plt.show()


# ─────────────────────────────────────────────
# 8. PREDICT ON CUSTOM REVIEWS
# ─────────────────────────────────────────────
word_index = imdb.get_word_index()

def encode_review(text: str) -> np.ndarray:
    """Tokenize and encode a raw string review."""
    tokens = text.lower().split()
    ids = [word_index.get(w, 2) + 3 for w in tokens]   # 0=pad,1=start,2=oov,3=unused
    ids = [id_ for id_ in ids if id_ < VOCAB_SIZE]
    return pad_sequences([ids], maxlen=MAX_LENGTH, padding="post", truncating="post")

def predict_sentiment(text: str) -> None:
    encoded = encode_review(text)
    score   = model.predict(encoded, verbose=0)[0][0]
    label   = "POSITIVE 😊" if score >= 0.5 else "NEGATIVE 😞"
    print(f"\nReview  : {text[:80]}...")
    print(f"Score   : {score:.4f}  →  {label}")

# Test examples
predict_sentiment(
    "This movie was absolutely brilliant. The acting was superb and the "
    "story kept me engaged throughout. A masterpiece of modern cinema."
)
predict_sentiment(
    "Terrible film. The plot made no sense and the acting was wooden. "
    "I fell asleep halfway through. Complete waste of time."
)
predict_sentiment(
    "The film had some interesting moments but overall felt rushed. "
    "The ending was disappointing though the visuals were impressive."
)

Loading IMDB dataset...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples : (25000, 200)
Test samples     : (25000, 200)
Positive (train) : 12500 / 25000


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training model...
Epoch 1/10
228/313 ━━━━━━━━━━━━━━━━━━━━ 1:28 1s/step - accuracy: 0.5053 - loss: 0.7290